# Impetus — A100 Optimized
## Lightning.ai | Baseline + Reranker on A100

A100 optimizations: Flash Attention 2, bfloat16 native, batch_size=64

In [ ]:
import sys, os, subprocess, torch

# Clone repo
if not os.path.exists("/content/Impetus"):
    subprocess.run(["git", "clone", "--depth", "1",
        "https://github.com/EdhieBM/Impetus.git", "/content/Impetus"])
os.chdir("/content/Impetus")
sys.path.insert(0, "/content/Impetus")

# Install deps (torch comes preinstalled on Lightning)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "datasets",
    "evaluate", "sentencepiece", "scipy", "pandas", "tqdm"])

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DTYPE = torch.bfloat16  # A100 native
BATCH_SIZE = 64  # A100 has 80GB, can go big

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded")

In [ ]:
from energy_module.scorers import MajorityVotingScorer
from benchmarks.run_reranker import batch_generate, extract_answer_number
from benchmarks.run_baseline import format_gsm8k_prompt, load_gsm8k

scorer = MajorityVotingScorer()
N_MAX = 20

ds = load_gsm8k("test", 50)
questions = [ex["question"] for ex in ds]
ref_answers = [extract_answer_number(ex["answer"]) for ex in ds]

# Baseline (greedy) — batch=64 on A100
print("Generating baselines...")
baseline_outputs = batch_generate(model, tokenizer,
    [format_gsm8k_prompt(q) for q in questions],
    do_sample=False, batch_size=BATCH_SIZE)
baseline_nums = [extract_answer_number(o) for o in baseline_outputs]

# 20 candidates per question
print(f"Generating {50 * N_MAX} candidates...")
all_prompts = []
q_indices = []
for i, q in enumerate(questions):
    p = format_gsm8k_prompt(q)
    for _ in range(N_MAX):
        all_prompts.append(p)
        q_indices.append(i)

all_outputs = batch_generate(model, tokenizer, all_prompts,
    do_sample=True, temperature=0.7, batch_size=BATCH_SIZE)

# Group by question
candidates_by_q = {i: [] for i in range(50)}
for idx, out in zip(q_indices, all_outputs):
    candidates_by_q[idx].append(out)

# Evaluate
baseline_acc = sum(1 for i in range(50) if baseline_nums[i] == ref_answers[i]) / 50
print("\n" + "="*50)
print(f"{'Method':<20} {'Accuracy':<10}")
print("-"*30)
print(f"{'Baseline':<20} {baseline_acc:.2%}")

for N in [5, 10, 20]:
    correct = 0
    for i in range(50):
        pool = candidates_by_q[i][:N]
        scores = scorer.score_batch(questions[i], pool)
        best = min(zip(pool, scores), key=lambda x: x[1])[0]
        if extract_answer_number(best) == ref_answers[i]:
            correct += 1
    acc = correct / 50
    print(f"{'N='+str(N):<20} {acc:.2%} (Δ{acc-baseline_acc:+.2%})")
print("="*50)